# Reto ETL: dos fuentes, una decisión
**Inteligencia de Negocios · Actividad autónoma individual**

## Pregunta de decisión
> Una biblioteca de barrio tiene presupuesto limitado. **¿Qué 10 libros del catálogo conviene comprar primero**, combinando la calidad percibida (calificación en estrellas del portal) con la trayectoria editorial del título (número de ediciones registradas en una API pública) en relación con su precio?

## Fuentes
| | Fuente | Formato | Qué aporta | Nivel de detalle |
|---|---|---|---|---|
| **A** | [books.toscrape.com](https://books.toscrape.com/) (portal de práctica para scraping) | HTML estático, con paginación y fichas por libro | título, precio, estrellas, UPC, categoría, stock | **una fila por libro** (aparece en el listado y en su ficha) |
| **B** | [Open Library Search API](https://openlibrary.org/dev/docs/api/search) | JSON | obras/ediciones asociadas a un título, nº de ediciones, valoraciones | **varias filas por título** (una por obra o edición agrupada) → requiere agrupar antes de integrar |

**Campo de relación:** el título normalizado (`clave`). La fuente A no trae ISBN, por eso la unión es por título y se controla con una validación de claves.

**Limitaciones:** books.toscrape.com es un sitio de práctica y sus precios son ficticios, así que la recomendación es ilustrativa del método. Open Library puede devolver obras distintas con títulos genéricos; por eso solo se aceptan coincidencias exactas del título normalizado.

## Flujo
`Extraer (requests + BeautifulSoup / JSON)` → `copias locales (datos/<escenario>/)` → `Transformar (pandas)` → `Cargar en PostgreSQL` → `Consulta SQL` → `Pruebas con dos escenarios modificados`.

## 0. Configuración y parámetros

In [1]:
import os, re, time, json, math, shutil, hashlib, unicodedata
from pathlib import Path
from urllib.parse import urljoin, urlparse, quote

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from sqlalchemy import create_engine, text, bindparam
from IPython.display import display

pd.set_option("display.max_colwidth", 55)
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)

# ---- Fuentes ----
URL_INICIO = "https://books.toscrape.com/"            # Fuente A: página inicial del portal
URL_B      = "https://openlibrary.org/search.json"    # Fuente B: API JSON pública
N_PAGINAS  = 5                                         # páginas del listado a recorrer (>= 3)
PAUSA_S    = 1.0                                       # pausa de cortesía entre peticiones reales
HEADERS    = {"User-Agent": "reto-etl-inteligencia-negocios/1.0 (actividad academica)"}

# ---- Carpeta de copias locales ----
RAIZ = Path(os.getenv("RETO_RAIZ", "datos"))

# ---- Parámetros de la regla de decisión (umbrales) ----
PARAMS = {
    "min_estrellas": 3,    # variable 1 (fuente A): calidad percibida mínima
    "min_ediciones": 5,   # variable 2 (fuente B): trayectoria editorial mínima
    "top_k": 10,           # tamaño de la lista final
}
print("Carpeta de datos:", RAIZ.resolve())
print("Parámetros:", PARAMS)

Carpeta de datos: C:\Users\Rebeca\Documents\UPSE\6to Semestre\Inteligencia de Negocios\Ciclo 2\reto_etl\datos
Parámetros: {'min_estrellas': 3, 'min_ediciones': 5, 'top_k': 10}


### Acceso a datos con copia local
`obtener(url, carpeta, online)` devuelve el contenido de una URL.
- Si ya existe la copia local en `carpeta`, **lee el archivo** (así los tres escenarios se reproducen sin internet).
- Si no existe y `online=True`, descarga, **guarda la copia** y espera `PAUSA_S` segundos.

In [2]:
def ruta_local(url, carpeta):
    """Convierte una URL en la ruta del archivo local donde se guarda su copia."""
    u = urlparse(url)
    if u.query:
        h = hashlib.sha1(u.query.encode("utf-8")).hexdigest()[:12]
        return Path(carpeta) / u.netloc / u.path.strip("/") / f"q_{h}.json"
    p = u.path.strip("/") or "index.html"
    return Path(carpeta) / u.netloc / p

URLS_CONSULTADAS = []   # registro para documentar qué se consultó

def obtener(url, carpeta, online=True):
    f = ruta_local(url, carpeta)
    URLS_CONSULTADAS.append(url)
    if f.exists():
        return f.read_bytes().decode("utf-8")
    if not online:
        raise FileNotFoundError(f"No hay copia local de {url} (se esperaba {f})")
    ultimo_error = None
    for intento in range(1, 5):
        try:
            r = requests.get(url, headers=HEADERS, timeout=(15, 60))
            r.raise_for_status()
            break
        except requests.RequestException as e:
            ultimo_error = e
            print(f"  intento {intento}/4 fallo ({type(e).__name__}); reintento en {5*intento}s...")
            time.sleep(5 * intento)
    else:
        raise ultimo_error
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_bytes(r.content)
    time.sleep(PAUSA_S)
    return r.content.decode("utf-8")

## 1. Extraer los datos

### Fuente A · Web scraping (HTML)
El código **no tiene URLs de libros escritas a mano**: parte de la página inicial, lee los enlaces de cada libro del listado y sigue el botón *next* para recorrer la paginación. Después visita la **ficha** de cada libro (UPC, categoría, stock).

Una fila de `a_raw` = **una aparición de un libro en el listado**, enriquecida con los datos de su ficha.

In [3]:
ESTRELLAS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def extraer_listado(carpeta, online=True):
    """Recorre la paginación desde la página inicial y devuelve un DataFrame con cada aparición de un libro."""
    filas, url = [], URL_INICIO
    for n in range(1, N_PAGINAS + 1):
        soup = BeautifulSoup(obtener(url, carpeta, online), "html.parser")
        for art in soup.select("article.product_pod"):
            a = art.select_one("h3 a")
            filas.append({
                "pagina": n,
                "titulo": a["title"],
                "url_ficha": urljoin(url, a["href"]),                     # enlace descubierto en el HTML
                "precio_gbp": float(re.search(r"\d+(?:\.\d+)?", art.select_one("p.price_color").text).group()),
                "estrellas": ESTRELLAS[art.select_one("p.star-rating")["class"][1]],
            })
        siguiente = soup.select_one("li.next a")
        if siguiente is None:
            break
        url = urljoin(url, siguiente["href"])
    return pd.DataFrame(filas)

def extraer_fichas(urls, carpeta, online=True):
    """Visita cada ficha (una vez por URL) y extrae UPC, categoría y stock."""
    filas = []
    for url in pd.unique(urls):
        soup = BeautifulSoup(obtener(url, carpeta, online), "html.parser")
        tabla = {tr.th.text.strip(): tr.td.text.strip() for tr in soup.select("table.table-striped tr")}
        categoria = soup.select("ul.breadcrumb li")[2].text.strip()
        m = re.search(r"(\d+)", tabla.get("Availability", ""))
        filas.append({"url_ficha": url, "upc": tabla["UPC"], "categoria": categoria,
                      "stock": int(m.group(1)) if m else 0})
    return pd.DataFrame(filas)

def extraer_a(carpeta, online=True):
    listado = extraer_listado(carpeta, online)
    fichas = extraer_fichas(listado["url_ficha"], carpeta, online)
    return listado.merge(fichas, on="url_ficha", how="left", validate="m:1")

In [4]:
BASE = RAIZ / "base"
URLS_CONSULTADAS.clear()
a_raw = extraer_a(BASE, online=True)       # primera ejecución: descarga y guarda copias en datos/base/
urls_a = list(dict.fromkeys(URLS_CONSULTADAS))
print(f"Filas extraídas de A: {len(a_raw)}  |  páginas del listado recorridas: {a_raw['pagina'].nunique()}  |  fichas visitadas: {a_raw['url_ficha'].nunique()}")
print("\nURL de las páginas del listado consultadas:")
for u in urls_a:
    if "/catalogue/page-" in u or u.rstrip("/") == URL_INICIO.rstrip("/"):
        print("  ", u)
print(f"\n(+ {a_raw['url_ficha'].nunique()} fichas individuales, p. ej.: {a_raw['url_ficha'].iloc[0]})")
display(a_raw.head(8))

Filas extraídas de A: 100  |  páginas del listado recorridas: 5  |  fichas visitadas: 100

URL de las páginas del listado consultadas:
   https://books.toscrape.com/
   https://books.toscrape.com/catalogue/page-2.html
   https://books.toscrape.com/catalogue/page-3.html
   https://books.toscrape.com/catalogue/page-4.html
   https://books.toscrape.com/catalogue/page-5.html

(+ 100 fichas individuales, p. ej.: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)


,pagina,titulo,url_ficha,precio_gbp,estrellas,upc,categoria,stock
0,1,A Light in the Attic,https://books.toscrape.com/catalogue/a-light-in-the...,51.77,3,a897fe39b1053632,Poetry,22
1,1,Tipping the Velvet,https://books.toscrape.com/catalogue/tipping-the-ve...,53.74,1,90fa61229261140a,Historical Fiction,20
2,1,Soumission,https://books.toscrape.com/catalogue/soumission_998...,50.10,1,6957f44c3847a760,Fiction,20
3,1,Sharp Objects,https://books.toscrape.com/catalogue/sharp-objects_...,47.82,4,e00eb4fd7b871a48,Mystery,20
4,1,Sapiens: A Brief History of Humankind,https://books.toscrape.com/catalogue/sapiens-a-brie...,54.23,5,4165285e1663650f,History,20
5,1,The Requiem Red,https://books.toscrape.com/catalogue/the-requiem-re...,22.65,1,f77dbf2323deb740,Young Adult,19
6,1,The Dirty Little Secrets of Getting Your Dream Job,https://books.toscrape.com/catalogue/the-dirty-litt...,33.34,4,2597b5a345f45e1b,Business,19
7,1,The Coming Woman: A Novel Based on the Life of the ...,https://books.toscrape.com/catalogue/the-coming-wom...,17.93,3,e72a5dfc7e9267b2,Default,19


### Fuente B · JSON (Open Library)
Para cada título de A se consulta `search.json?title=...`. La API devuelve **varios documentos por título** (obras o ediciones agrupadas por autor), por eso una fila de `b_raw` = **un documento devuelto por la API para una consulta**. Cada respuesta JSON se guarda como copia local.

In [5]:
CAMPOS_B = "key,title,author_name,first_publish_year,edition_count,ratings_average,ratings_count"

def normalizar_titulo(t):
    """Clave de unión: minúsculas, sin acentos, sin subtítulo/paréntesis, sin puntuación ni artículo inicial."""
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().lower()
    t = re.split(r"[:(]", t)[0]
    t = t.replace("'", "")
    t = re.sub(r"[^a-z0-9]+", " ", t).strip()
    t = re.sub(r"^(the|a|an) ", "", t)
    return t

def extraer_b(titulos, carpeta, online=True):
    filas = []
    for titulo in pd.unique(titulos):
        url = f"{URL_B}?title={quote(titulo)}&limit=10&fields={CAMPOS_B}"
        datos = json.loads(obtener(url, carpeta, online))
        for d in datos.get("docs", []):
            filas.append({
                "clave_consulta": normalizar_titulo(titulo),
                "titulo_consultado": titulo,
                "ol_key": d.get("key"),
                "ol_titulo": d.get("title"),
                "autor": ", ".join(d.get("author_name", [])[:2]) or None,
                "anio_primera_pub": d.get("first_publish_year"),
                "ediciones": d.get("edition_count"),
                "ratings_avg": d.get("ratings_average"),
                "ratings_cnt": d.get("ratings_count"),
            })
    return pd.DataFrame(filas, columns=["clave_consulta","titulo_consultado","ol_key","ol_titulo","autor",
                                        "anio_primera_pub","ediciones","ratings_avg","ratings_cnt"])

In [6]:
URLS_CONSULTADAS.clear()
b_raw = extraer_b(a_raw["titulo"], BASE, online=True)
urls_b = list(dict.fromkeys(URLS_CONSULTADAS))
print(f"Consultas a la API: {len(urls_b)}  |  documentos JSON devueltos (filas de b_raw): {len(b_raw)}")
print("Ejemplo de URL consultada:\n  ", urls_b[0])
display(b_raw.head(8))
print("\nDocumentos devueltos por consulta (describe):")
print(b_raw.groupby("clave_consulta").size().describe().round(2).to_string())

Consultas a la API: 100  |  documentos JSON devueltos (filas de b_raw): 311
Ejemplo de URL consultada:
   https://openlibrary.org/search.json?title=A%20Light%20in%20the%20Attic&limit=10&fields=key,title,author_name,first_publish_year,edition_count,ratings_average,ratings_count


,clave_consulta,titulo_consultado,ol_key,ol_titulo,autor,anio_primera_pub,ediciones,ratings_avg,ratings_cnt
0,light in the attic,A Light in the Attic,/works/OL3368239W,A Light in the Attic,"Shel Silverstein, S. . Silverstein",1981.0,27,0.000000,0.0
1,light in the attic,A Light in the Attic,/works/OL4461060W,A ghost-light in the attic,"Thomson, Pat, Pat Thomson",1995.0,2,4.000000,1.0
2,light in the attic,A Light in the Attic,/works/OL6466271W,A Ghost Light in the Attic (Flashbacks),Pat Thomson,1999.0,2,4.000000,1.0
3,light in the attic,A Light in the Attic,/works/OL35348434W,Poems & Drawings Where the Sidewalk Ends A Light in...,Shel Silverstein,1974.0,1,NaN,NaN
4,tipping the velvet,Tipping the Velvet,/works/OL551012W,Tipping the Velvet,"Sarah Waters, Sarah Waters",1998.0,22,3.846154,13.0
5,soumission,Soumission,/works/OL17106654W,Soumission,Michel Houellebecq,2015.0,21,3.266667,15.0
6,soumission,Soumission,/works/OL32513272W,Soumission,Amina Niang,2022.0,3,NaN,NaN
7,soumission,Soumission,/works/OL25837079W,Pouvoir et Soumission,"Caralyn Knight, Seth Daniels",2013.0,1,0.000000,0.0



Documentos devueltos por consulta (describe):
count    45.00
mean      6.91
std       3.85
min       1.00
25%       3.00
50%      10.00
75%      10.00
max      10.00


## 2. Transformar y tomar una decisión

### Granularidad de cada fuente
| | Qué representa una fila |
|---|---|
| `a_raw` | una **aparición** de un libro en el listado (puede repetirse) |
| `b_raw` | un **documento de Open Library** devuelto para una consulta (varios por título) |
| **resultado** | **un libro** (UPC único) con su indicador, su cumplimiento de la regla y su prioridad |

### Normalización y deduplicación
- **Clave de unión (`clave`)**: título normalizado (minúsculas, sin acentos, sin subtítulo ni puntuación, sin artículo inicial) aplicado en A y en B.
- **Clave de duplicado en A = `upc`**. El UPC identifica el producto de forma única; el **título solo no sirve**, porque dos libros legítimos distintos pueden compartir título (otra edición, otro autor). Usar el título habría eliminado registros legítimos; el UPC no.
- En B se conservan únicamente los documentos cuyo título normalizado **coincide exactamente** con la clave consultada; luego se **agrupa por clave** (suma de ediciones, nº de obras, valoraciones) para tener una fila por título antes de unir.

In [7]:
def aplicar_regla(df, p):
    """Regla de decisión con dos variables + prioridad por indicador. Los umbrales llegan por parámetro."""
    d = df.copy()
    d["cumple"] = (d["coincide"]
                   & (d["estrellas"] >= p["min_estrellas"])
                   & (d["ediciones_ol"].fillna(-1) >= p["min_ediciones"]))
    elegibles = d[d["cumple"]].sort_values(["ediciones_por_libra", "upc"], ascending=[False, True])
    d["prioridad"] = (pd.Series(range(1, len(elegibles) + 1), index=elegibles.index)
                        .reindex(d.index).astype("Int64"))
    pr = d["prioridad"].fillna(10**9)
    d["decision"] = np.where(pr <= p["top_k"], "comprar", np.where(d["cumple"], "lista_espera", "descartar"))
    d["motivo"] = np.select(
        [~d["coincide"], d["estrellas"] < p["min_estrellas"], d["ediciones_ol"].fillna(-1) < p["min_ediciones"]],
        ["sin_coincidencia_ol", "estrellas_bajo_umbral", "ediciones_bajo_umbral"], default="")
    return d

def transformar(a_raw, b_raw, params):
    st = {}
    # ---------- A: normalizar y deduplicar ----------
    a = a_raw.copy()
    a["clave"] = a["titulo"].map(normalizar_titulo)
    st["filas_a_crudas"] = len(a)
    dup = a.duplicated("upc", keep="first")
    st["duplicados_a"] = int(dup.sum())
    st["duplicados_con_precio_distinto"] = int((a.groupby("upc")["precio_gbp"].nunique() > 1).sum())
    a_dedup = a[~dup].reset_index(drop=True)
    st["filas_a_dedup"] = len(a_dedup)
    st["libros_con_titulo_repetido_y_upc_distinto"] = int(a_dedup.duplicated("clave", keep=False).sum())

    # ---------- B: normalizar, filtrar coincidencias exactas y agrupar ----------
    b = b_raw.copy()
    b["clave"] = b["ol_titulo"].map(normalizar_titulo)
    st["docs_b_crudos"] = len(b)
    b_ok = b[b["clave"] == b["clave_consulta"]].copy()
    st["docs_b_con_titulo_exacto"] = len(b_ok)
    b_ok["ediciones"] = pd.to_numeric(b_ok["ediciones"], errors="coerce")
    b_ok["ratings_cnt"] = pd.to_numeric(b_ok["ratings_cnt"], errors="coerce")
    b_ok["ratings_avg"] = pd.to_numeric(b_ok["ratings_avg"], errors="coerce")
    b_grp = (b_ok.groupby("clave", as_index=False)
                 .agg(obras_ol=("ol_key", "nunique"),
                      ediciones_ol=("ediciones", lambda s: s.sum(min_count=1)),
                      ratings_total=("ratings_cnt", "sum"),
                      ratings_avg_ol=("ratings_avg", "mean")))
    st["claves_b_agrupadas"] = len(b_grp)
    assert b_grp["clave"].is_unique, "B agrupada debe tener una fila por clave"

    # ---------- Integración ----------
    j = a_dedup.merge(b_grp, on="clave", how="left", validate="m:1")
    assert len(j) == len(a_dedup), "La unión multiplicó filas"
    assert j["upc"].is_unique, "UPC repetido tras la unión"
    j["coincide"] = j["obras_ol"].notna()
    st["filas_integradas"] = len(j)
    st["con_coincidencia"] = int(j["coincide"].sum())
    st["sin_coincidencia"] = int((~j["coincide"]).sum())

    # ---------- Indicador derivado ----------
    j["ediciones_por_libra"] = (j["ediciones_ol"] / j["precio_gbp"]).round(6)
    final = aplicar_regla(j, params)
    cols = ["clave", "upc", "titulo", "categoria", "precio_gbp", "estrellas", "stock", "obras_ol", "ediciones_ol",
            "ratings_total", "ratings_avg_ol", "ediciones_por_libra", "coincide", "cumple", "prioridad", "decision", "motivo"]
    final = final[cols]
    for c in ["obras_ol", "ediciones_ol", "ratings_total"]:
        final[c] = final[c].astype("Int64")
    return {"a_dedup": a_dedup, "b_agrupada": b_grp, "integrado": j.drop(columns=["pagina", "url_ficha"]), "final": final, "stats": st}

In [8]:
T = transformar(a_raw, b_raw, PARAMS)
st = T["stats"]
resumen = pd.Series(st, name="valor").to_frame()
display(resumen)

print(f"""
Duplicados detectados en A (mismo UPC): {st['duplicados_a']}  ->  se conserva la primera aparición y se descartan las demás.
Duplicados con precio distinto (conflicto): {st['duplicados_con_precio_distinto']}
Libros con título repetido pero UPC distinto (NO se eliminan, son registros legítimos): {st['libros_con_titulo_repetido_y_upc_distinto']}
Sin coincidencia en Open Library: {st['sin_coincidencia']} de {st['filas_integradas']} libros
  -> se CONSERVAN en la tabla con coincide=False y motivo='sin_coincidencia_ol'; no puede evaluarse la variable de ediciones, por lo que quedan fuera de la regla (decision='descartar').
""")

# Validación de claves: la unión no multiplicó filas
assert T["b_agrupada"]["clave"].is_unique
assert len(T["integrado"]) == len(T["a_dedup"]) == st["filas_a_crudas"] - st["duplicados_a"]
assert T["integrado"]["upc"].is_unique
print("Validación OK: B agrupada tiene 1 fila por clave; filas tras la unión =", len(T["integrado"]),
      "= filas de A sin duplicados =", len(T["a_dedup"]))
display(T["integrado"][T["integrado"]["coincide"]].head(8))

,valor
filas_a_crudas,100
duplicados_a,0
duplicados_con_precio_distinto,0
filas_a_dedup,100
libros_con_titulo_repetido_y_upc_distinto,0
docs_b_crudos,311
docs_b_con_titulo_exacto,133
claves_b_agrupadas,42
filas_integradas,100
con_coincidencia,42



Duplicados detectados en A (mismo UPC): 0  ->  se conserva la primera aparición y se descartan las demás.
Duplicados con precio distinto (conflicto): 0
Libros con título repetido pero UPC distinto (NO se eliminan, son registros legítimos): 0
Sin coincidencia en Open Library: 58 de 100 libros
  -> se CONSERVAN en la tabla con coincide=False y motivo='sin_coincidencia_ol'; no puede evaluarse la variable de ediciones, por lo que quedan fuera de la regla (decision='descartar').

Validación OK: B agrupada tiene 1 fila por clave; filas tras la unión = 100 = filas de A sin duplicados = 100


,titulo,precio_gbp,estrellas,upc,categoria,stock,clave,obras_ol,ediciones_ol,ratings_total,ratings_avg_ol,coincide,ediciones_por_libra
0,A Light in the Attic,51.77,3,a897fe39b1053632,Poetry,22,light in the attic,1.0,27.0,0.0,0.000000,True,0.521538
1,Tipping the Velvet,53.74,1,90fa61229261140a,Historical Fiction,20,tipping the velvet,1.0,22.0,13.0,3.846154,True,0.409378
2,Soumission,50.10,1,6957f44c3847a760,Fiction,20,soumission,4.0,26.0,15.0,3.266667,True,0.518962
3,Sharp Objects,47.82,4,e00eb4fd7b871a48,Mystery,20,sharp objects,2.0,36.0,36.0,1.902778,True,0.752823
5,The Requiem Red,22.65,1,f77dbf2323deb740,Young Adult,19,requiem red,1.0,2.0,0.0,0.000000,True,0.088300
6,The Dirty Little Secrets of Getting Your Dream Job,33.34,4,2597b5a345f45e1b,Business,19,dirty little secrets of getting your dream job,1.0,2.0,0.0,NaN,True,0.059988
9,The Black Maria,52.15,1,1dfe412b8ac00530,Poetry,19,black maria,4.0,4.0,1.0,2.000000,True,0.076702
11,Shakespeare's Sonnets,20.66,4,30a7f60cd76ca58c,Poetry,19,shakespeares sonnets,5.0,22.0,0.0,0.000000,True,1.064860


In [9]:
# Mismos libros con título repetido (si existen): se muestran para evidenciar que el UPC los distingue
rep = T["a_dedup"][T["a_dedup"].duplicated("clave", keep=False)].sort_values("clave")[["clave", "titulo", "upc", "precio_gbp", "categoria"]]
print("Libros con la misma clave de título pero distinto UPC (se conservan):", len(rep))
display(rep.head(10))

Libros con la misma clave de título pero distinto UPC (se conservan): 0


,clave,titulo,upc,precio_gbp,categoria


### Indicador y regla de decisión
- **Indicador derivado:** `ediciones_por_libra = ediciones_ol / precio_gbp` → cuánta trayectoria editorial se obtiene por cada libra gastada.
- **Regla con dos variables (de fuentes distintas):** el libro es candidato si `estrellas >= min_estrellas` (fuente A) **y** `ediciones_ol >= min_ediciones` (fuente B).
- **Prioridad:** los candidatos se ordenan por `ediciones_por_libra` (desc.; desempate por `upc`) y se recomiendan los primeros `top_k`.

In [10]:
f = T["final"]
ed = f["ediciones_ol"].dropna().astype(float)
print("Distribución de ediciones_ol (libros con coincidencia):")
print(ed.describe(percentiles=[.25, .5, .75, .9]).round(1).to_string())
print("\nDistribución de estrellas (A):")
print(f["estrellas"].value_counts().sort_index().to_string())

for var, val in [("min_ediciones", PARAMS["min_ediciones"])]:
    pct = (ed < val).mean() * 100
    print(f"\nJustificación de {var}={val}: deja fuera al {pct:.0f}% de los títulos con coincidencia (los de menor trayectoria editorial) y conserva al {100-pct:.0f}% restante.")
pct_e = (f["estrellas"] < PARAMS["min_estrellas"]).mean() * 100
print(f"Justificación de min_estrellas={PARAMS['min_estrellas']}: exige 4 o más estrellas (nota alta en una escala de 5); excluye al {pct_e:.0f}% de los libros.")
print(f"\nLibros que cumplen la regla: {int(f['cumple'].sum())}  |  recomendados (top {PARAMS['top_k']}): {int((f['decision']=='comprar').sum())}")
cols_top = ["prioridad", "titulo", "precio_gbp", "estrellas", "ediciones_ol", "ediciones_por_libra", "decision"]
display(f[f["decision"] == "comprar"].sort_values("prioridad")[cols_top])

Distribución de ediciones_ol (libros con coincidencia):
count    42.0
mean     11.1
std      12.4
min       1.0
25%       2.2
50%       6.0
75%      14.8
90%      26.9
max      57.0

Distribución de estrellas (A):
estrellas
1    22
2    19
3    22
4    18
5    19

Justificación de min_ediciones=5: deja fuera al 36% de los títulos con coincidencia (los de menor trayectoria editorial) y conserva al 64% restante.
Justificación de min_estrellas=3: exige 4 o más estrellas (nota alta en una escala de 5); excluye al 41% de los libros.

Libros que cumplen la regla: 16  |  recomendados (top 10): 10


,prioridad,titulo,precio_gbp,estrellas,ediciones_ol,ediciones_por_libra,decision
84,1,Patience,10.16,3,57,5.610236,comprar
11,2,Shakespeare's Sonnets,20.66,4,22,1.064860,comprar
12,3,Set Me Free,17.46,5,15,0.859107,comprar
3,4,Sharp Objects,47.82,4,36,0.752823,comprar
38,5,Behind Closed Doors,52.22,4,39,0.746840,comprar
54,6,Thirst,17.27,5,11,0.636943,comprar
0,7,A Light in the Attic,51.77,3,27,0.521538,comprar
34,8,Sophie's World,15.94,5,8,0.501882,comprar
32,9,The Elephant Tree,23.82,5,7,0.293871,comprar
14,10,Rip it Up and Start Again,35.02,5,10,0.285551,comprar


**Sensibilidad:** se varía un umbral (`min_ediciones`) manteniendo el resto fijo y se compara cuántos candidatos hay y qué libros entran o salen de la lista respecto a los parámetros base.

In [11]:
base_top = set(aplicar_regla(T["integrado"], PARAMS).query("decision == 'comprar'")["upc"])
titulos = T["integrado"].set_index("upc")["titulo"]
filas = []
for me in [5, 10, 20, 40]:
    r = aplicar_regla(T["integrado"], {**PARAMS, "min_ediciones": me})
    top = set(r.query("decision == 'comprar'")["upc"])
    filas.append({"min_ediciones": me, "candidatos": int(r["cumple"].sum()), "en_top": len(top),
                  "entran_vs_base": "; ".join(sorted(titulos[u] for u in top - base_top)) or "-",
                  "salen_vs_base": "; ".join(sorted(titulos[u] for u in base_top - top)) or "-"})
display(pd.DataFrame(filas))

,min_ediciones,candidatos,en_top,entran_vs_base,salen_vs_base
0,5,16,10,-,-
1,10,9,9,Wall and Piece,Sophie's World; The Elephant Tree
2,20,5,5,-,Rip it Up and Start Again; Set Me Free; Sophie's Wo...
3,40,1,1,-,A Light in the Attic; Behind Closed Doors; Rip it U...


## 3. Cargar y verificar en PostgreSQL
La conexión se configura con variables de entorno (por defecto, las del `docker-compose.yml` incluido: usuario `reto`, base `retoetl`, puerto `5432`). **No hay credenciales personales en el notebook.**

La tabla `decision_libros` guarda el resultado transformado con la columna `escenario` (clave primaria `(escenario, upc)`). La carga es idempotente: borra y vuelve a insertar solo el escenario que se está cargando.

In [12]:
def crear_engine():
    url = os.getenv("DATABASE_URL")
    if not url:
        url = "postgresql+psycopg2://{u}:{p}@{h}:{port}/{d}".format(
            u=os.getenv("PGUSER", "reto"), p=os.getenv("PGPASSWORD", "reto"),
            h=os.getenv("PGHOST", "localhost"), port=os.getenv("PGPORT", "5432"),
            d=os.getenv("PGDATABASE", "retoetl"))
    return create_engine(url)

engine = crear_engine()
with engine.connect() as con:
    print(con.execute(text("select version()")).scalar())

DDL = """
CREATE TABLE IF NOT EXISTS decision_libros (
    escenario           text        NOT NULL,
    clave               text        NOT NULL,
    upc                 text        NOT NULL,
    titulo              text        NOT NULL,
    categoria           text,
    precio_gbp          numeric(8,2) NOT NULL,
    estrellas           smallint    NOT NULL,
    stock               integer,
    obras_ol            integer,
    ediciones_ol        integer,
    ratings_total       integer,
    ratings_avg_ol      double precision,
    ediciones_por_libra numeric(14,6),
    coincide            boolean     NOT NULL,
    cumple              boolean     NOT NULL,
    prioridad           integer,
    decision            text        NOT NULL,
    motivo              text,
    PRIMARY KEY (escenario, upc)
)"""

def cargar(final, escenario):
    d = final.copy()
    d.insert(0, "escenario", escenario)
    with engine.begin() as con:
        con.execute(text(DDL))
        con.execute(text("DELETE FROM decision_libros WHERE escenario = :e"), {"e": escenario})
        d.to_sql("decision_libros", con, if_exists="append", index=False, method="multi", chunksize=500)
    with engine.connect() as con:
        n = con.execute(text("SELECT count(*) FROM decision_libros WHERE escenario = :e"), {"e": escenario}).scalar()
    assert n == len(final), f"Se esperaban {len(final)} filas y quedaron {n}"
    return n

n = cargar(T["final"], "base")
print(f"Filas cargadas en PostgreSQL para el escenario 'base': {n}")

PostgreSQL 16.16 (Debian 16.16-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit
Filas cargadas en PostgreSQL para el escenario 'base': 100


### Consulta SQL de verificación
La consulta **no lee los resultados de Python**: recalcula el indicador y aplica la regla (dos umbrales como parámetros) sobre las columnas realmente cargadas en `decision_libros`, y numera la prioridad por escenario con `ROW_NUMBER()`.

In [13]:
SQL_DECISION = text("""
WITH elegibles AS (
    SELECT escenario, upc, titulo, precio_gbp, estrellas, ediciones_ol,
           ROUND(ediciones_ol / precio_gbp, 6) AS indicador
    FROM decision_libros
    WHERE escenario IN :escenarios
      AND coincide
      AND estrellas     >= :min_estrellas
      AND ediciones_ol  >= :min_ediciones
), ordenados AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY escenario ORDER BY indicador DESC, upc) AS prioridad
    FROM elegibles
)
SELECT escenario, prioridad, upc, titulo, precio_gbp, estrellas, ediciones_ol, indicador
FROM ordenados
WHERE prioridad <= :top_k
ORDER BY escenario, prioridad
""").bindparams(bindparam("escenarios", expanding=True))

def consulta_sql(escenarios, params=PARAMS):
    with engine.connect() as con:
        return pd.read_sql(SQL_DECISION, con, params={"escenarios": list(escenarios), **params})

sql_base = consulta_sql(["base"])
display(sql_base)

# La salida SQL debe coincidir con la decisión calculada en Python
py_top = T["final"].query("decision == 'comprar'").sort_values("prioridad")["upc"].tolist()
assert sql_base["upc"].tolist() == py_top, "SQL y Python no coinciden"
print("OK: el orden de prioridad de SQL coincide con el de Python para el escenario base.")

,escenario,prioridad,upc,titulo,precio_gbp,estrellas,ediciones_ol,indicador
0,base,1,9429b4d59c537af5,Patience,10.16,3,57,5.610236
1,base,2,30a7f60cd76ca58c,Shakespeare's Sonnets,20.66,4,22,1.064860
2,base,3,ce6396b0f23f6ecc,Set Me Free,17.46,5,15,0.859107
3,base,4,e00eb4fd7b871a48,Sharp Objects,47.82,4,36,0.752823
4,base,5,be5cc846f45496fb,Behind Closed Doors,52.22,4,39,0.746840
5,base,6,709822d0b5bcb7f4,Thirst,17.27,5,11,0.636943
6,base,7,a897fe39b1053632,A Light in the Attic,51.77,3,27,0.521538
7,base,8,6be3beb0793a53e7,Sophie's World,15.94,5,8,0.501882
8,base,9,5dada2b7be26bd03,The Elephant Tree,23.82,5,7,0.293871
9,base,10,a34ba96d4081e6a4,Rip it Up and Start Again,35.02,5,10,0.285551


OK: el orden de prioridad de SQL coincide con el de Python para el escenario base.


## 4. Demostrar el comportamiento con dos pruebas

Se define `ejecutar_etl(escenario, carpeta)`, que repite **el mismo proceso** (extracción desde las copias locales → transformación → carga en PostgreSQL) para cualquier carpeta de datos. Primero se comprueba que el escenario base se reproduce sin internet y da exactamente el mismo resultado.

In [14]:
def ejecutar_etl(escenario, carpeta, params=PARAMS):
    a = extraer_a(carpeta, online=False)                       # solo copias locales
    b = extraer_b(a["titulo"], carpeta, online=False)
    t = transformar(a, b, params)
    cargar(t["final"], escenario)
    return t

res_base = ejecutar_etl("base", BASE)
pd.testing.assert_frame_equal(res_base["final"], T["final"])
print("Escenario base reproducido desde copias locales: resultado idéntico al de la primera ejecución.")
tabla_base = res_base["final"]
cand = tabla_base[tabla_base["cumple"]].sort_values("prioridad")
K = PARAMS["top_k"]
print(f"Candidatos en base: {len(cand)} (se necesitan al menos {K + 1} para la prueba 1)")
assert len(cand) > K, "Baje min_ediciones o min_estrellas: se necesitan más de top_k candidatos"

Escenario base reproducido desde copias locales: resultado idéntico al de la primera ejecución.
Candidatos en base: 16 (se necesitan al menos 11 para la prueba 1)


### Utilidades para crear los escenarios (copia independiente del original + edición del HTML)
Cada escenario parte de `shutil.copytree` de `datos/base`; **el original nunca se modifica**. Los cambios se hacen sobre el bloque `<li>` del libro dentro del HTML del listado.

In [15]:
def copiar_base(nombre):
    destino = RAIZ / nombre
    if destino.exists():
        shutil.rmtree(destino)
    shutil.copytree(BASE, destino)
    return destino

def localizar_bloque(carpeta, url_ficha):
    """Devuelve (archivo, texto, inicio, fin) del <li> del listado que contiene al libro.
    Solo busca en las páginas del listado, no en las fichas individuales."""
    slug = url_ficha.rstrip("/").split("/")[-2]
    sitio = carpeta / urlparse(URL_INICIO).netloc
    paginas = [sitio / "index.html"] + sorted((sitio / "catalogue").glob("page-*.html"))
    for f in paginas:
        if not f.exists():
            continue
        txt = f.read_text(encoding="utf-8")
        enlace = f"{slug}/index.html"
        if enlace in txt:
            i = txt.index(enlace)
            ini, fin = txt.rfind("<li", 0, i), txt.index("</li>", i) + len("</li>")
            return f, txt, ini, fin
    raise LookupError(f"No se encontró {slug} en el HTML del listado")

def editar_precio(carpeta, url_ficha, nuevo_precio):
    f, txt, ini, fin = localizar_bloque(carpeta, url_ficha)
    bloque = txt[ini:fin]
    nuevo = re.sub(r'(<p class="price_color">)[^<]*(</p>)', lambda m: f"{m.group(1)}£{nuevo_precio:.2f}{m.group(2)}", bloque)
    assert nuevo != bloque
    f.write_text(txt[:ini] + nuevo + txt[fin:], encoding="utf-8")
    return f

def duplicar_registro(carpeta, url_ficha):
    f, txt, ini, fin = localizar_bloque(carpeta, url_ficha)
    f.write_text(txt[:fin] + "\n" + txt[ini:fin] + txt[fin:], encoding="utf-8")
    return f

def comparar(antes, despues):
    """Compara dos listas ordenadas de UPC: entran, salen y cambian de posición."""
    pa = {u: i + 1 for i, u in enumerate(antes)}
    pd_ = {u: i + 1 for i, u in enumerate(despues)}
    return ({u for u in pd_ if u not in pa}, {u for u in pa if u not in pd_},
            {u: (pa[u], pd_[u]) for u in pa if u in pd_ and pa[u] != pd_[u]})

### Prueba 1 · Cambio que **altera** la decisión
**Edición:** se sube el precio del libro que ocupa la prioridad 1 en el listado HTML de la copia local, de modo que su indicador caiga por debajo del libro que hoy está en la posición `top_k + 1`.

In [16]:
obj = cand.iloc[0]               # prioridad 1 en el escenario base
ref = cand.iloc[K]               # primer libro que hoy queda fuera de la lista (prioridad K+1)
url_obj = a_raw.loc[a_raw["titulo"] == obj["titulo"], "url_ficha"].iloc[0]

factor = math.ceil(2 * obj["ediciones_por_libra"] / ref["ediciones_por_libra"]) + 1
precio_nuevo = round(float(obj["precio_gbp"]) * factor, 2)

print(f"Registro editado: «{obj['titulo']}» (UPC {obj['upc']})")
print(f"  precio original £{obj['precio_gbp']:.2f}  ->  precio editado £{precio_nuevo:.2f}  (x{factor})")

# ---------- CÁLCULO A MANO (solo con valores de la tabla base, sin ejecutar el ETL) ----------
ind_esperado = obj["ediciones_ol"] / precio_nuevo
print(f"\nCálculo manual del indicador: {obj['ediciones_ol']} ediciones / £{precio_nuevo:.2f} = {ind_esperado:.6f}")
print(f"  (antes: {obj['ediciones_ol']} / £{obj['precio_gbp']:.2f} = {obj['ediciones_por_libra']:.6f}; "
      f"el libro K+1 «{ref['titulo']}» tiene {ref['ediciones_por_libra']:.6f})")

manual = [(-(ind_esperado if r.upc == obj["upc"] else r.ediciones_por_libra), r.upc) for r in cand.itertuples()]
manual.sort()
esperado_top = [u for _, u in manual[:K]]
pos_esperada_obj = [u for _, u in manual].index(obj["upc"]) + 1
base_top_list = cand["upc"].tolist()[:K]
ent_e, sal_e, mov_e = comparar(base_top_list, esperado_top)

print(f"\nPREDICCIÓN")
print(f"  - «{obj['titulo']}» pasa de la prioridad 1 a la {pos_esperada_obj} -> {'SALE' if obj['upc'] not in esperado_top else 'baja pero permanece'} de la lista.")
print(f"  - Entra: {[cand.set_index('upc').loc[u,'titulo'] for u in ent_e]}")
print(f"  - Los demás libros de la lista suben de posición: {len(mov_e)} cambian de lugar.")
assert obj["upc"] not in esperado_top and ref["upc"] in esperado_top

Registro editado: «Patience» (UPC 9429b4d59c537af5)
  precio original £10.16  ->  precio editado £457.20  (x45)

Cálculo manual del indicador: 57 ediciones / £457.20 = 0.124672
  (antes: 57 / £10.16 = 5.610236; el libro K+1 «This One Summer» tiene 0.256542)

PREDICCIÓN
  - «Patience» pasa de la prioridad 1 a la 15 -> SALE de la lista.
  - Entra: ['This One Summer']
  - Los demás libros de la lista suben de posición: 9 cambian de lugar.


In [17]:
ESC1 = "esc_cambia_decision"
d1 = copiar_base(ESC1)
archivo = editar_precio(d1, url_obj, precio_nuevo)
print("Archivo HTML modificado:", archivo.relative_to(RAIZ))

res1 = ejecutar_etl(ESC1, d1)
f1 = res1["final"]
sql1 = consulta_sql(["base", ESC1])
top1_sql = sql1[sql1["escenario"] == ESC1]["upc"].tolist()
top1_py = f1.query("decision == 'comprar'").sort_values("prioridad")["upc"].tolist()

# ---------- ASERCIONES ----------
fila = f1[f1["upc"] == obj["upc"]].iloc[0]
assert float(fila["precio_gbp"]) == precio_nuevo, "El ETL no leyó el precio editado"
assert abs(float(fila["ediciones_por_libra"]) - ind_esperado) < 1e-5, "El indicador observado no coincide con el cálculo manual"
assert top1_py == esperado_top, "La lista observada (Python) difiere de la esperada"
assert top1_sql == esperado_top, "La lista observada (SQL) difiere de la esperada"
assert obj["upc"] not in top1_sql and ref["upc"] in top1_sql
assert res1["stats"]["duplicados_a"] == 0
print("Aserciones OK: indicador, lista (Python y SQL) y entrada/salida coinciden con la predicción.")

ent, sal, mov = comparar(sql_base["upc"].tolist(), top1_sql)
nombre = titulos_all = pd.concat([sql_base, sql1]).drop_duplicates("upc").set_index("upc")["titulo"]
print(f"\nRESULTADO OBSERVADO (consulta SQL, base vs {ESC1}):")
print("  Entran :", [nombre[u] for u in ent])
print("  Salen  :", [nombre[u] for u in sal])
print("  Cambian de posición:", {nombre[u]: f"{a}->{b}" for u, (a, b) in mov.items()})
print(f"""
EXPLICACIÓN: se cambió solo el precio de «{obj['titulo']}». Como el indicador es ediciones/precio, al multiplicar el precio por {factor} su indicador pasó de {obj['ediciones_por_libra']:.4f} a {ind_esperado:.4f},
por debajo del de «{ref['titulo']}» ({ref['ediciones_por_libra']:.4f}), que era el primero fuera de la lista. Las variables de la regla (estrellas y ediciones) no cambiaron, por eso el libro sigue siendo candidato pero ya no está entre los {K} primeros.""")
display(sql1.pivot(index="upc", columns="escenario", values="prioridad").join(nombre).sort_values(ESC1))

Archivo HTML modificado: esc_cambia_decision\books.toscrape.com\catalogue\page-5.html
Aserciones OK: indicador, lista (Python y SQL) y entrada/salida coinciden con la predicción.

RESULTADO OBSERVADO (consulta SQL, base vs esc_cambia_decision):
  Entran : ['This One Summer']
  Salen  : ['Patience']
  Cambian de posición: {"Shakespeare's Sonnets": '2->1', 'Set Me Free': '3->2', 'Sharp Objects': '4->3', 'Behind Closed Doors': '5->4', 'Thirst': '6->5', 'A Light in the Attic': '7->6', "Sophie's World": '8->7', 'The Elephant Tree': '9->8', 'Rip it Up and Start Again': '10->9'}

EXPLICACIÓN: se cambió solo el precio de «Patience». Como el indicador es ediciones/precio, al multiplicar el precio por 45 su indicador pasó de 5.6102 a 0.1247,
por debajo del de «This One Summer» (0.2565), que era el primero fuera de la lista. Las variables de la regla (estrellas y ediciones) no cambiaron, por eso el libro sigue siendo candidato pero ya no está entre los 10 primeros.


,base,esc_cambia_decision,titulo
upc,,,
30a7f60cd76ca58c,2.0,1.0,Shakespeare's Sonnets
ce6396b0f23f6ecc,3.0,2.0,Set Me Free
e00eb4fd7b871a48,4.0,3.0,Sharp Objects
be5cc846f45496fb,5.0,4.0,Behind Closed Doors
709822d0b5bcb7f4,6.0,5.0,Thirst
a897fe39b1053632,7.0,6.0,A Light in the Attic
6be3beb0793a53e7,8.0,7.0,Sophie's World
5dada2b7be26bd03,9.0,8.0,The Elephant Tree
a34ba96d4081e6a4,10.0,9.0,Rip it Up and Start Again


### Prueba 2 · Cambio que **conserva** la decisión
**Edición:** se duplica un registro real (el libro en la prioridad 3 de la base) copiando su bloque `<li>` dentro del HTML del listado de una copia independiente del original. Es un duplicado exacto (mismo UPC), no un libro nuevo.

In [18]:
dup_row = cand.iloc[2]
url_dup = a_raw.loc[a_raw["titulo"] == dup_row["titulo"], "url_ficha"].iloc[0]
print(f"Registro duplicado: «{dup_row['titulo']}» (UPC {dup_row['upc']}), prioridad base = {dup_row['prioridad']}, indicador = {dup_row['ediciones_por_libra']:.6f}")
print(f"""
PREDICCIÓN
  - El proceso detecta exactamente 1 duplicado (filas crudas de A: {len(a_raw)} -> {len(a_raw) + 1}; tras deduplicar vuelven a {len(a_raw)}).
  - Como el duplicado se elimina por UPC ANTES de agrupar e integrar, no se altera precio, estrellas ni ediciones:
    el indicador ({dup_row['ediciones_por_libra']:.6f}) y la prioridad ({dup_row['prioridad']}) deben permanecer idénticos y la lista de {K} libros no cambia.""")

Registro duplicado: «Set Me Free» (UPC ce6396b0f23f6ecc), prioridad base = 3, indicador = 0.859107

PREDICCIÓN
  - El proceso detecta exactamente 1 duplicado (filas crudas de A: 100 -> 101; tras deduplicar vuelven a 100).
  - Como el duplicado se elimina por UPC ANTES de agrupar e integrar, no se altera precio, estrellas ni ediciones:
    el indicador (0.859107) y la prioridad (3) deben permanecer idénticos y la lista de 10 libros no cambia.


In [19]:
ESC2 = "esc_duplicado"
d2 = copiar_base(ESC2)
archivo2 = duplicar_registro(d2, url_dup)
print("Archivo HTML modificado:", archivo2.relative_to(RAIZ))

res2 = ejecutar_etl(ESC2, d2)
f2, s2 = res2["final"], res2["stats"]

# ---------- ASERCIONES ----------
assert s2["filas_a_crudas"] == len(a_raw) + 1, "El HTML duplicado no produjo una fila extra"
assert s2["duplicados_a"] == 1, "El proceso debe detectar exactamente 1 duplicado"
assert s2["filas_a_dedup"] == res_base["stats"]["filas_a_dedup"]
fb = tabla_base[tabla_base["upc"] == dup_row["upc"]].iloc[0]
fd = f2[f2["upc"] == dup_row["upc"]].iloc[0]
assert fd["ediciones_por_libra"] == fb["ediciones_por_libra"], "El indicador cambió"
assert fd["prioridad"] == fb["prioridad"] and fd["decision"] == fb["decision"], "La clasificación cambió"
pd.testing.assert_frame_equal(f2, tabla_base)      # tabla completa idéntica
sql2 = consulta_sql(["base", ESC2])
assert sql2[sql2.escenario == "base"]["upc"].tolist() == sql2[sql2.escenario == ESC2]["upc"].tolist()
print("Aserciones OK: duplicado detectado y tabla/indicador/clasificación idénticos al escenario base.")

ent2, sal2, mov2 = comparar(sql2[sql2.escenario == "base"]["upc"].tolist(), sql2[sql2.escenario == ESC2]["upc"].tolist())
print(f"\nRESULTADO OBSERVADO (SQL, base vs {ESC2}): entran={len(ent2)}, salen={len(sal2)}, cambian de posición={len(mov2)}")
print(f"Duplicados detectados: {s2['duplicados_a']}  |  filas A crudas -> deduplicadas: {s2['filas_a_crudas']} -> {s2['filas_a_dedup']}")
print(f"""
EXPLICACIÓN: el registro duplicado compartía UPC con el original, de modo que la deduplicación por UPC lo eliminó antes de la integración. Por eso la unión con B no multiplicó filas
y «{dup_row['titulo']}» conserva indicador {fd['ediciones_por_libra']:.6f}, prioridad {fd['prioridad']} y decisión '{fd['decision']}'. (Si se hubiera deduplicado por título, o no se hubiera deduplicado, el conteo y el indicador agregado podrían haberse distorsionado.)""")
display(pd.DataFrame({
    "valor": ["precio_gbp", "estrellas", "ediciones_ol", "ediciones_por_libra", "prioridad", "decision"],
    "base": [fb[c] for c in ["precio_gbp", "estrellas", "ediciones_ol", "ediciones_por_libra", "prioridad", "decision"]],
    ESC2:   [fd[c] for c in ["precio_gbp", "estrellas", "ediciones_ol", "ediciones_por_libra", "prioridad", "decision"]]}))

Archivo HTML modificado: esc_duplicado\books.toscrape.com\index.html
Aserciones OK: duplicado detectado y tabla/indicador/clasificación idénticos al escenario base.

RESULTADO OBSERVADO (SQL, base vs esc_duplicado): entran=0, salen=0, cambian de posición=0
Duplicados detectados: 1  |  filas A crudas -> deduplicadas: 101 -> 100

EXPLICACIÓN: el registro duplicado compartía UPC con el original, de modo que la deduplicación por UPC lo eliminó antes de la integración. Por eso la unión con B no multiplicó filas
y «Set Me Free» conserva indicador 0.859107, prioridad 3 y decisión 'comprar'. (Si se hubiera deduplicado por título, o no se hubiera deduplicado, el conteo y el indicador agregado podrían haberse distorsionado.)


,valor,base,esc_duplicado
0,precio_gbp,17.46,17.46
1,estrellas,5,5
2,ediciones_ol,15,15
3,ediciones_por_libra,0.859107,0.859107
4,prioridad,3,3
5,decision,comprar,comprar


### Comparación final de los tres escenarios (consulta SQL sobre lo cargado)

In [20]:
sql_all = consulta_sql(["base", ESC1, ESC2])
cmp_ = sql_all.pivot(index="upc", columns="escenario", values="prioridad").join(
    sql_all.drop_duplicates("upc").set_index("upc")[["titulo"]])
display(cmp_[["titulo", "base", ESC1, ESC2]].sort_values("base", na_position="last"))

with engine.connect() as con:
    conteo = pd.read_sql(text("SELECT escenario, count(*) AS filas, count(*) FILTER (WHERE decision='comprar') AS comprar FROM decision_libros GROUP BY escenario ORDER BY escenario"), con)
display(conteo)

,titulo,base,esc_cambia_decision,esc_duplicado
upc,,,,
9429b4d59c537af5,Patience,1.0,NaN,1.0
30a7f60cd76ca58c,Shakespeare's Sonnets,2.0,1.0,2.0
ce6396b0f23f6ecc,Set Me Free,3.0,2.0,3.0
e00eb4fd7b871a48,Sharp Objects,4.0,3.0,4.0
be5cc846f45496fb,Behind Closed Doors,5.0,4.0,5.0
709822d0b5bcb7f4,Thirst,6.0,5.0,6.0
a897fe39b1053632,A Light in the Attic,7.0,6.0,7.0
6be3beb0793a53e7,Sophie's World,8.0,7.0,8.0
5dada2b7be26bd03,The Elephant Tree,9.0,8.0,9.0


,escenario,filas,comprar
0,base,100,10
1,esc_cambia_decision,100,10
2,esc_duplicado,100,10


## 5. Conclusión

In [21]:
top = sql_base
costo = float(top["precio_gbp"].sum())
print(f"""DECISIÓN (escenario base, regla: estrellas >= {PARAMS['min_estrellas']} y ediciones >= {PARAMS['min_ediciones']}, prioridad por ediciones por libra, top {PARAMS['top_k']}):
Se recomienda comprar {len(top)} libros por un costo total de £{costo:.2f}.
Primero en la lista: «{top.iloc[0]['titulo']}» con {top.iloc[0]['ediciones_ol']} ediciones a £{float(top.iloc[0]['precio_gbp']):.2f} = {float(top.iloc[0]['indicador']):.2f} ediciones por libra.
Último de la lista: «{top.iloc[-1]['titulo']}» con {float(top.iloc[-1]['indicador']):.2f} ediciones por libra.
Candidatos que cumplen la regla: {int(tabla_base['cumple'].sum())} de {len(tabla_base)} libros; sin coincidencia en la fuente B: {int((~tabla_base['coincide']).sum())}.

ROBUSTEZ:
- Prueba 1 (precio editado de «{obj['titulo']}»): la decisión CAMBIÓ; salió de la prioridad 1 y entró «{nombre[ref['upc']]}».
- Prueba 2 (registro duplicado de «{dup_row['titulo']}»): la decisión se CONSERVÓ; el proceso detectó {s2['duplicados_a']} duplicado y el indicador ({fd['ediciones_por_libra']:.6f}) y la prioridad ({fd['prioridad']}) no variaron.
""")

DECISIÓN (escenario base, regla: estrellas >= 3 y ediciones >= 5, prioridad por ediciones por libra, top 10):
Se recomienda comprar 10 libros por un costo total de £292.14.
Primero en la lista: «Patience» con 57 ediciones a £10.16 = 5.61 ediciones por libra.
Último de la lista: «Rip it Up and Start Again» con 0.29 ediciones por libra.
Candidatos que cumplen la regla: 16 de 100 libros; sin coincidencia en la fuente B: 58.

ROBUSTEZ:
- Prueba 1 (precio editado de «Patience»): la decisión CAMBIÓ; salió de la prioridad 1 y entró «This One Summer».
- Prueba 2 (registro duplicado de «Set Me Free»): la decisión se CONSERVÓ; el proceso detectó 1 duplicado y el indicador (0.859107) y la prioridad (3) no variaron.



> **Para completar con sus propias palabras:** interprete las cifras de la salida anterior (¿qué tan sensible es la lista al umbral de ediciones?, ¿qué limitaciones tiene unir por título?, ¿qué haría con los libros sin coincidencia?) y ajuste la justificación de los umbrales si cambia sus valores. ## Conclusión

**Decisión.** Con la regla de estrellas ≥ 3 (fuente A) y ediciones en Open Library ≥ 5 (fuente B), solo 16 de los 100 libros analizados resultaron candidatos. De ellos se recomienda comprar los 10 con mayor indicador de ediciones por libra, con un costo total de £292.14. El primero de la lista es «Patience», con 57 ediciones a £10.16, es decir 5.61 ediciones por libra. El último es «Rip it Up and Start Again», con 0.29 ediciones por libra.

**Umbrales.** Los umbrales iniciales (4 estrellas y 10 ediciones) dejaban solo 7 candidatos, menos de los 11 necesarios para comprobar que un libro puede entrar o salir de la lista de 10. Por eso los bajé a 3 estrellas y 5 ediciones, que mantienen un filtro mínimo de calidad percibida y de trayectoria editorial sin dejar la lista sin opciones.

**Pruebas.**
- *Prueba 1 (la decisión cambia):* al subir el precio de «Patience» en el HTML local, su indicador (ediciones ÷ precio) cayó por debajo del libro que estaba en la posición 11, «This One Summer». Resultado: «Patience» salió de la lista y «This One Summer» entró; los libros intermedios subieron una posición. Las variables de la regla (estrellas y ediciones) no cambiaron, solo el precio, que interviene en el indicador.
- *Prueba 2 (la decisión se conserva):* al duplicar el registro de «Set Me Free», el proceso detectó 1 duplicado por UPC y lo eliminó antes de integrar las fuentes. Su indicador (0.859107) y su prioridad (3) no variaron, y la lista de 10 libros quedó idéntica a la del escenario base.

**Limitaciones.** La unión entre fuentes es por título porque el portal no trae ISBN; por eso solo acepté coincidencias exactas del título normalizado, y puede haber falsos positivos con títulos genéricos. 58 de los 100 libros no tuvieron coincidencia en Open Library; los conservé en la tabla marcados como «sin_coincidencia_ol», pero no pudieron evaluarse, así que la lista final solo considera a los 42 libros con datos en ambas fuentes. Además, los precios de books.toscrape.com son ficticios y el número de ediciones mide trayectoria editorial, no demanda real de los lectores, por lo que el resultado ilustra el método más que una recomendación de compra real.